In [ ]:
import os

import pandas as pd
import janitor

from utilities import walk_through_files

ALTERNATE_DUMP_PATH = os.path.join("raw-data", "datadump")
SAVEPATH = os.path.join("intermediate-output", "device-home-summary.csv.gz")

In [ ]:
file_lists = tuple(walk_through_files(ALTERNATE_DUMP_PATH, "csv.gz", fullpath=True))

df_raw = pd.concat(
    [
        pd.read_csv(f, compression="gzip", usecols=["device_xid", "area_id", "date"])
        for f in file_lists
    ]
)

df_counts = (
    df_raw
    .groupby(["device_xid", "area_id"])
    .size()
    .reset_index()
    .rename_column(0, "count")
)

df_distinct_days = (
    df_raw
    .groupby("device_xid")["date"]
    .nunique()
    .rename("distinct_days")
    .reset_index()
)

In [ ]:
df_ranked = (
    df_counts
    .sort_values(["device_xid", "count"], ascending=[True, False])
    .assign(rank=lambda x: x.groupby("device_xid").cumcount())
)

df_modal = (
    df_ranked
    .query("rank == 0")
    .rename(columns={"area_id": "modal_area_id", "count": "modal_count"})
    [["device_xid", "modal_area_id", "modal_count"]]
)

df_second = (
    df_ranked
    .query("rank == 1")
    .rename(columns={"count": "second_count"})
    [["device_xid", "second_count"]]
)

df_summary = (
    df_modal
    .merge(df_second, on="device_xid", how="left")
    .fillna({"second_count": 0})
    .merge(df_distinct_days, on="device_xid")
    .assign(
        modal_share=lambda x: x["modal_count"] / x["distinct_days"],
        gap=lambda x: (x["modal_count"] - x["second_count"]) / x["distinct_days"],
    )
)

df_summary.to_csv(SAVEPATH, index=False, compression="gzip")
df_summary.head()